<img src="https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/unifor-logo.png" width="400">
<br>
<b>
<font size="6" face="arial" color="blue">
    Graduação em Ciência da Computação
</font>
</b>
<br>
<b>
<font size="4" face="arial">
    Disciplina: Métodos Quantitativos em Computação
</font>
</b>

**Orientador: Prof. Me. Ricardo Carubbi** <br>
*Docente da Graduação e Pós-Graduação em Ciência de Dados e Inteligência Artificial*<br>
*Laboratório de Ciência de Dados e Inteligência Artificial*<br>
*Universidade de Fortaleza*<br>

# **Aula 5 — Tipos, qualidade e pré-processamento básico**



## 1. Objetivos de aprendizagem

Ao final da aula, o aluno será capaz de:

- distinguir unidade de análise, observação, variável, identificador e metadado;
- classificar variáveis por sua natureza estatística e seu papel analítico, considerando significado, processo de obtenção e granularidade do registro;
- diferenciar tipos estatísticos de tipos computacionais e avaliar a coerência de conversões;
- diagnosticar completude, validade, consistência e unicidade com denominadores e chaves adequados;
- justificar transformações básicas e preservar a base bruta.



## 2. Agenda

1. Unidade de análise e papel das variáveis — 15 min
2. Natureza estatística das variáveis — 25 min
3. Tipo estatístico versus tipo computacional — 15 min
4. Qualidade dos dados orientada pela pergunta — 20 min
5. Pré-processamento dos dados — 20 min
6. Estudo e exercícios — 5 min

## 3. Unidade de análise e papel das variáveis



### 3.1 Unidade de análise, observação e variável

Antes de classificar uma coluna, é necessário definir o que cada linha representa. A **unidade de análise** é a entidade sobre a qual se deseja produzir uma descrição ou conclusão. Uma **observação** é o registro dessa unidade na base, enquanto uma **variável** é uma característica que pode assumir valores diferentes entre as observações. Em uma matriz de dados organizada, as linhas representam casos e as colunas representam variáveis (BARBETTA; REIS; BORNIA, 2010, pp. 52–53).

No conjunto Palmer Penguins, o significado da linha depende do protocolo de coleta. Não basta dizer que uma linha é “um pinguim”: é preciso verificar se o mesmo indivíduo pode aparecer em campanhas diferentes e quais campos identificam cada registro. Essa definição orientará posteriormente a escolha das chaves e a análise de duplicidades.

Uma coluna pode funcionar como identificador, distinguindo entidades ou registros, ou como metadado, descrevendo o contexto da coleta. Esses papéis dependem da pergunta: uma data de coleta, por exemplo, também pode ser analisada para investigar diferenças entre períodos.

![Matriz de dados dos azulejos: unidades ou casos nas linhas e variáveis nas colunas.](../assets/imgs/u1_a05/barbetta_matriz_azulejos.png)

**Figura 1 — Organização de uma matriz de dados.** As linhas correspondem aos azulejos observados; as colunas registram características, com unidades explícitas nas medidas quantitativas. Fonte: esquema original da seção 3.1 (BARBETTA; REIS; BORNIA, 2010, p. 53).



### 3.2 Processo de obtenção, unidade, precisão e granularidade

A classificação estatística depende do significado e do processo de obtenção. É necessário perguntar se o valor foi contado, medido, codificado ou agrupado; qual é sua unidade; qual precisão foi registrada; e qual é a granularidade do registro. Uma idade registrada apenas em anos aparece como inteiro, embora o tempo seja conceitualmente contínuo. Da mesma forma, uma massa medida em gramas continua sendo quantitativa contínua mesmo quando o instrumento registra valores arredondados (PINHEIRO et al., 2009, pp. 24–26).

**Aplicação guiada:** para cada coluna selecionada do Palmer Penguins, responda: o que representa, como foi obtida, qual é sua unidade ou domínio e qual papel exerce na análise?

In [227]:
# Carrega a base bruta sem aplicar transformações
import pandas as pd

url = "https://raw.githubusercontent.com/carubbi/MQ/main/data/raw/penguins_raw.csv"
penguins = pd.read_csv(url)
penguins.head()

,studyName,Sample Number,Species,Region,Island,Stage,Individual ID,Clutch Completion,Date Egg,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,Delta 15 N (o/oo),Delta 13 C (o/oo),Comments
0,PAL0708,1,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A1,Yes,2007-11-11,39.1,18.7,181.0,3750.0,MALE,NaN,NaN,Not enough blood for isotopes.
1,PAL0708,2,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A2,Yes,2007-11-11,39.5,17.4,186.0,3800.0,FEMALE,8.94956,-24.69454,NaN
2,PAL0708,3,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A1,Yes,2007-11-16,40.3,18.0,195.0,3250.0,FEMALE,8.36821,-25.33302,NaN
3,PAL0708,4,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A2,Yes,2007-11-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Adult not sampled.
4,PAL0708,5,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N3A1,Yes,2007-11-16,36.7,19.3,193.0,3450.0,FEMALE,8.76651,-25.32426,NaN


**Tabela 1 - Primeiras observações da base bruta.** Cada linha registra uma observação da coleta, enquanto as colunas combinam características do pinguim, identificadores e metadados. A inspeção inicial ajuda a formular perguntas, mas não substitui o diagnóstico do conjunto completo. Fonte: Palmer Penguins.

In [228]:
# Resume armazenamento, preenchimento e exemplos de cada coluna
inventario = pd.DataFrame({
    "variavel": penguins.columns,
    "dtype_inicial": penguins.dtypes.astype(str).to_numpy(),
    "nao_ausentes": penguins.notna().sum().to_numpy(),
    "ausentes": penguins.isna().sum().to_numpy(),
    "valores_distintos": penguins.nunique(dropna=True).to_numpy(),
})
inventario

,variavel,dtype_inicial,nao_ausentes,ausentes,valores_distintos
0,studyName,str,344,0,3
1,Sample Number,int64,344,0,152
2,Species,str,344,0,3
3,Region,str,344,0,1
4,Island,str,344,0,3
5,Stage,str,344,0,1
6,Individual ID,str,344,0,190
7,Clutch Completion,str,344,0,2
8,Date Egg,str,344,0,50
9,Culmen Length (mm),float64,342,2,164


**Tabela 2 - Inventário estrutural do Palmer Penguins.** A base contém 344 observações e 17 colunas com diferentes quantidades de valores ausentes e distintos. Os tipos exibidos descrevem o armazenamento inicial; a natureza estatística ainda depende do significado e do processo de obtenção de cada variável.

## 4. Natureza estatística das variáveis



### 4.1 Qualitativas nominais e binárias

Variáveis **qualitativas nominais** registram categorias sem ordem natural. Espécie e ilha são exemplos: seus valores podem ser comparados por igualdade e resumidos por frequências, mas não por diferenças numéricas. Códigos numéricos usados para representar categorias continuam nominais quando os números funcionam apenas como rótulos.

Uma variável **binária** possui duas categorias e constitui um caso particular das variáveis qualitativas. O destaque é útil porque respostas como presença/ausência ou sim/não aparecem com frequência, mas os códigos 0 e 1 não transformam automaticamente essas categorias em medidas quantitativas (MORETTIN; BUSSAB, 2010, pp. 26–27).



### 4.2 Qualitativas ordinais

Variáveis **qualitativas ordinais** possuem categorias com ordem natural, como baixo, médio e alto. A ordem permite comparações de posição, porém não garante que a distância entre categorias consecutivas seja mensurável ou constante. A ordenação deve decorrer do significado da variável e, quando usada no software, precisa ser declarada explicitamente.



### 4.3 Quantitativas discretas e contínuas

Variáveis **quantitativas discretas** são produzidas tipicamente por contagem e assumem valores separados, como o número de ocorrências. Variáveis **quantitativas contínuas** resultam de mensuração e podem, conceitualmente, assumir qualquer valor em um intervalo, como massa ou comprimento. A precisão finita do instrumento não altera essa natureza (MORETTIN; BUSSAB, 2010, pp. 26–27).

![Classificação das variáveis em qualitativas nominais e ordinais e quantitativas discretas e contínuas.](../assets/imgs/u1_a05/tipos_dados.png)

**Figura 2 — Classificação das variáveis.** O esquema distingue variáveis qualitativas nominais e ordinais e quantitativas discretas e contínuas. A classificação depende do significado da variável, não do tipo de armazenamento. Fonte: CARUBBI, 2026.



### 4.4 Operações e representações coerentes

O tipo estatístico delimita operações interpretáveis. Frequências e proporções são adequadas para categorias; ordenações são pertinentes para categorias ordinais; diferenças, médias e medidas de dispersão exigem valores quantitativos com significado numérico. Na Aula 6, essa classificação orientará a escolha entre tabelas de frequências e representações gráficas.

| Variável | Natureza estatística | Operações coerentes |
| --- | --- | --- |
| `Species` | qualitativa nominal | frequências, proporções e comparação de categorias |
| `Sex` | qualitativa binária no registro disponível | frequências e proporções, considerando ausências |
| `Sample Number` | identificador sequencial | unicidade e rastreamento; não média |
| `Body Mass (g)` | quantitativa contínua | diferenças, medidas-resumo e histogramas |
| `Date Egg` | temporal | ordenação, intervalos e agrupamentos temporais |

In [229]:
# Contrasta aparência, armazenamento e papel analítico
cols_sel = [
    "Species",
    "Individual ID",
    "Sample Number",
    "Body Mass (g)",
    "Date Egg",
]
perfil = pd.DataFrame({
    "variavel": cols_sel,
    "dtype": [str(penguins[coluna].dtype) for coluna in cols_sel],
    "exemplo": [penguins[coluna].dropna().iloc[0] for coluna in cols_sel],
    "distintos": [penguins[coluna].nunique() for coluna in cols_sel],
})
perfil

,variavel,dtype,exemplo,distintos
0,Species,str,Adelie Penguin (Pygoscelis adeliae),3
1,Individual ID,str,N1A1,190
2,Sample Number,int64,1,152
3,Body Mass (g),float64,3750.0,94
4,Date Egg,str,2007-11-11,50


**Tabela 3 - Aparência e armazenamento de cinco variáveis.** `Sample Number` aparece como inteiro, mas funciona como identificador sequencial; `Date Egg` chega como texto, embora represente uma data. Esses contrastes mostram por que o `dtype` não determina sozinho a classificação estatística.

In [230]:
# Aplica operações compatíveis com o significado das variáveis
operacoes = pd.DataFrame([
    {
        "variavel": "Species",
        "operacao": "frequência da categoria mais comum",
        "resultado": int(penguins["Species"].value_counts().iloc[0]),
    },
    {
        "variavel": "Individual ID",
        "operacao": "quantidade de identificadores distintos",
        "resultado": int(penguins["Individual ID"].nunique()),
    },
    {
        "variavel": "Body Mass (g)",
        "operacao": "massa média observada, em gramas",
        "resultado": round(penguins["Body Mass (g)"].mean(), 2),
    },
])
operacoes

,variavel,operacao,resultado
0,Species,frequência da categoria mais comum,152.00
1,Individual ID,quantidade de identificadores distintos,190.00
2,Body Mass (g),"massa média observada, em gramas",4201.75


**Tabela 4 - Operações coerentes com o significado das variáveis.** Frequências descrevem espécies, a contagem de valores distintos auxilia a avaliação de identificadores e a média possui interpretação física para massa corporal. A mesma média não seria informativa para códigos ou identificadores.

## 5. Tipo estatístico versus tipo computacional



### 5.1 Significado estatístico e `dtype`

O **tipo estatístico** descreve o significado da variável e as operações que fazem sentido. O **tipo computacional** informa como o software armazena e processa os valores. Eles se relacionam, mas não existe correspondência biunívoca: uma coluna inteira pode conter códigos nominais; uma contagem com ausências pode ser armazenada como ponto flutuante; e uma data pode chegar como texto. A tipagem auxilia o processamento, mas não substitui a interpretação conceitual (BRUCE; BRUCE, 2019, pp. 23–26).



### 5.2 Categorias, números, datas e valores ausentes

Em pandas, `object` frequentemente armazena texto, mas também pode reunir conteúdos heterogêneos. `category` explicita um domínio categórico; categorias ordinais precisam ter sua ordem definida. Tipos inteiros e de ponto flutuante representam números, enquanto `datetime64` representa instantes ou datas. Valores ausentes podem modificar o `dtype` observado sem modificar a natureza estatística da variável.

Assim, a classificação deve seguir esta ordem: compreender o significado, examinar os valores e o processo de obtenção, definir a natureza estatística e somente então avaliar se o tipo computacional é adequado.



### 5.3 Conversões com controles antes e depois

Uma conversão não é correta apenas porque o comando foi executado sem erro. Antes de converter, registre número de linhas, ausências, domínios e formatos problemáticos. Depois, repita os controles e compare os resultados. A conversão deve preservar o significado e tornar explícitas eventuais perdas.

Para datas, valores que não puderem ser interpretados precisam ser identificados; para categorias, níveis e frequências devem ser preservados; para números, valores inválidos introduzidos pela conversão devem ser contabilizados. Esse procedimento separa uma transformação tecnicamente possível de uma decisão analiticamente justificável.

In [231]:
# Converte a data em uma cópia e compara as perdas
dados = penguins.copy(deep=True)
ausentes_antes = int(dados["Date Egg"].isna().sum())
dados["Date Egg"] = pd.to_datetime(dados["Date Egg"], errors="coerce")
controle_data = pd.DataFrame({
    "controle": ["linhas", "ausentes antes", "ausentes depois", "data mínima", "data máxima"],
    "resultado": [
        len(dados),
        ausentes_antes,
        int(dados["Date Egg"].isna().sum()),
        dados["Date Egg"].min().date(),
        dados["Date Egg"].max().date(),
    ],
})
controle_data

,controle,resultado
0,linhas,344
1,ausentes antes,0
2,ausentes depois,0
3,data mínima,2007-11-09
4,data máxima,2009-12-01


**Tabela 5 - Controle da conversão de `Date Egg`.** A conversão preserva as 344 observações, não introduz datas ausentes e recupera o intervalo de 2007 a 2009. O controle evidencia a mudança de representação computacional sem perda observada.

In [232]:
# Converte categorias e verifica níveis, frequências e ausências
cols_cat = ["Species", "Island", "Sex"]
controle_cat = []
for coluna in cols_cat:
    freq_antes = dados[coluna].value_counts(dropna=False).sort_index().to_dict()
    ausentes_antes = int(dados[coluna].isna().sum())
    dados[coluna] = dados[coluna].astype("category")
    freq_depois = dados[coluna].value_counts(dropna=False).sort_index().to_dict()
    controle_cat.append({
        "variavel": coluna,
        "dtype_depois": str(dados[coluna].dtype),
        "ausentes_preservados": ausentes_antes == int(dados[coluna].isna().sum()),
        "frequencias_preservadas": freq_antes == freq_depois,
    })
pd.DataFrame(controle_cat)

,variavel,dtype_depois,ausentes_preservados,frequencias_preservadas
0,Species,category,True,True
1,Island,category,True,True
2,Sex,category,True,True


**Tabela 6 - Controle das conversões categóricas.** Espécie, ilha e sexo passam a usar o tipo `category` sem alterar frequências nem ausências. O resultado confirma a preservação dos valores observados, não uma mudança em sua natureza estatística.

## 6. Qualidade dos dados orientada pela pergunta

A qualidade não é uma propriedade absoluta do arquivo. Ela deve ser avaliada em relação à pergunta, às variáveis necessárias e às regras conhecidas da coleta. A análise exploratória permite localizar dados incompletos, inválidos ou inconsistentes, mas verificações internas não provam que uma medição corresponde exatamente ao fenômeno real.



### 6.1 Completude e denominador analítico

**Completude** indica se os valores necessários estão disponíveis. Para a variável $j$, a proporção de ausências pode ser escrita como

$$
p_{	ext{aus},j}=\frac{m_j}{n},
$$

em que $m_j$ é o número de registros ausentes e $n$ é o total de registros considerado. O denominador deve ser declarado, pois pode representar a base completa ou um subconjunto analítico. Um **caso completo** depende do conjunto de variáveis exigido pela pergunta; portanto, não existe uma única contagem de casos completos válida para todas as análises.

Ausência, zero e uma categoria substantiva são situações distintas. Excluir ou preencher valores altera o conjunto analisado e pode modificar sua composição; nesta aula, o diagnóstico não autoriza imputação automática (MCKINNEY, 2017, pp. 209–214).



### 6.2 Validade e consistência

**Validade** verifica se um valor respeita domínio, formato, unidade e intervalo definidos. **Consistência** examina relações entre campos, como datas em ordem possível ou medidas compatíveis com regras documentadas. Um valor dentro do intervalo esperado pode ainda estar incorreto; por isso, o resultado do teste confirma somente a regra avaliada.



### 6.3 Unicidade, unidade de análise e chaves

**Unicidade** exige uma chave coerente com a unidade de análise. Uma linha integralmente repetida é diferente de um identificador que reaparece legitimamente em campanhas distintas. A pergunta correta não é apenas “há valores repetidos?”, mas “quais colunas deveriam identificar unicamente cada observação neste processo de coleta?”.

**Aplicação guiada:** formule uma pergunta envolvendo espécie, massa corporal e sexo. Em seguida, defina as variáveis necessárias, o denominador da completude, as regras de validade e a chave usada para examinar unicidade.

In [233]:
# Calcula quantidade e proporção de ausências por variável
ausencias = (
    pd.DataFrame({
        "ausentes": penguins.isna().sum(),
        "proporcao": penguins.isna().mean(),
    })
    .sort_values(["ausentes", "proporcao"], ascending=False)
)
ausencias

,ausentes,proporcao
Comments,290,0.843023
Delta 15 N (o/oo),14,0.040698
Delta 13 C (o/oo),13,0.037791
Sex,11,0.031977
Culmen Length (mm),2,0.005814
Culmen Depth (mm),2,0.005814
Flipper Length (mm),2,0.005814
Body Mass (g),2,0.005814
studyName,0,0.000000
Sample Number,0,0.000000


**Tabela 7 - Diagnóstico de completude por variável.** `Comments` possui 290 ausências em 344 registros, enquanto sexo apresenta 11 e algumas medidas corporais apresentam 2. A relevância dessas lacunas depende das variáveis exigidas pela pergunta analítica.

In [234]:
# Avalia completude no subconjunto exigido pela pergunta
cols_analise = ["Species", "Body Mass (g)", "Sex"]
completos = penguins[cols_analise].notna().all(axis=1)
resumo = pd.DataFrame({
    "medida": ["total de registros", "casos completos", "casos incompletos", "proporção completa"],
    "resultado": [
        len(penguins),
        int(completos.sum()),
        int((~completos).sum()),
        round(completos.mean(), 4),
    ],
})
resumo

,medida,resultado
0,total de registros,344.000
1,casos completos,333.000
2,casos incompletos,11.000
3,proporção completa,0.968


**Tabela 8 - Completude do subconjunto analítico.** Há 333 registros com espécie, massa corporal e sexo preenchidos, entre 344 observações. A contagem de casos completos depende das variáveis exigidas pela análise.

## 7. Pré-processamento

### 7.1 Importação e preservação da base bruta

Cada transformação será aplicada a uma cópia. A base bruta permitirá conferir as decisões e recuperar identificadores e metadados que não entram na análise.

In [235]:
# Importando a bilioteca de maipulação de dataframes
import pandas as pd
print(pd.__version__)

3.0.5


In [236]:
# Definir o endereço e exibir as cinco primeiras observações
url = '/content/penguins_raw.csv'
url = 'https://raw.githubusercontent.com/allisonhorst/palmerpenguins/master/inst/extdata/penguins_raw.csv'
df_raw = pd.read_csv(url)
df_raw.head()

,studyName,Sample Number,Species,Region,Island,Stage,Individual ID,Clutch Completion,Date Egg,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,Delta 15 N (o/oo),Delta 13 C (o/oo),Comments
0,PAL0708,1,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A1,Yes,2007-11-11,39.1,18.7,181.0,3750.0,MALE,NaN,NaN,Not enough blood for isotopes.
1,PAL0708,2,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A2,Yes,2007-11-11,39.5,17.4,186.0,3800.0,FEMALE,8.94956,-24.69454,NaN
2,PAL0708,3,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A1,Yes,2007-11-16,40.3,18.0,195.0,3250.0,FEMALE,8.36821,-25.33302,NaN
3,PAL0708,4,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A2,Yes,2007-11-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Adult not sampled.
4,PAL0708,5,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N3A1,Yes,2007-11-16,36.7,19.3,193.0,3450.0,FEMALE,8.76651,-25.32426,NaN


In [237]:
# Verificando informações dos dados
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   studyName            344 non-null    str    
 1   Sample Number        344 non-null    int64  
 2   Species              344 non-null    str    
 3   Region               344 non-null    str    
 4   Island               344 non-null    str    
 5   Stage                344 non-null    str    
 6   Individual ID        344 non-null    str    
 7   Clutch Completion    344 non-null    str    
 8   Date Egg             344 non-null    str    
 9   Culmen Length (mm)   342 non-null    float64
 10  Culmen Depth (mm)    342 non-null    float64
 11  Flipper Length (mm)  342 non-null    float64
 12  Body Mass (g)        342 non-null    float64
 13  Sex                  333 non-null    str    
 14  Delta 15 N (o/oo)    330 non-null    float64
 15  Delta 13 C (o/oo)    331 non-null    float64
 16  C

In [238]:
# Informando as dimensões
print(df_raw.shape)

(344, 17)


### 7.2 Seleção de variáveis e padronização

Selecione as colunas e padronize seus nomes, preservando as unidades das medidas. As medidas de comprimento estão em milímetros e a massa está em gramas. A padronização altera os nomes das colunas, sem converter essas unidades. Os identificadores continuam disponíveis na base bruta.

In [239]:
# Filtrando das colunas desejadas
cols = df_raw.columns.to_list()
cols_sel = cols[2:5] + cols[8:-3]
df = df_raw[cols_sel].copy()
df.head()

,Species,Region,Island,Date Egg,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex
0,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,2007-11-11,39.1,18.7,181.0,3750.0,MALE
1,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,2007-11-11,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,2007-11-16,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,2007-11-16,NaN,NaN,NaN,NaN,NaN
4,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,2007-11-16,36.7,19.3,193.0,3450.0,FEMALE


In [240]:
# Padronizar os nomes das colunas
cols_limpas = []

for coluna in cols_sel:
    nome = coluna.split(' (')[0]
    nome = nome.replace(' ', '_')
    nome = nome.upper()
    cols_limpas.append(nome)

df.columns = cols_limpas

df.head()

,SPECIES,REGION,ISLAND,DATE_EGG,CULMEN_LENGTH,CULMEN_DEPTH,FLIPPER_LENGTH,BODY_MASS,SEX
0,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,2007-11-11,39.1,18.7,181.0,3750.0,MALE
1,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,2007-11-11,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,2007-11-16,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,2007-11-16,NaN,NaN,NaN,NaN,NaN
4,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,2007-11-16,36.7,19.3,193.0,3450.0,FEMALE


A coluna `ID` combina a campanha e o identificador individual. Ela será exportada com os dados para permitir identificar cada registro. O caractere `_` separa as duas informações.

In [241]:
# Criar um identificador combinando campanha e indivíduo
df['ID'] = df_raw['studyName'] + '_' + df_raw['Individual ID']
df['ID'].head()

0    PAL0708_N1A1
1    PAL0708_N1A2
2    PAL0708_N2A1
3    PAL0708_N2A2
4    PAL0708_N3A1
Name: ID, dtype: str

Abreviar os nomes das espécies e uniformizar a escrita de sexo altera a representação das categorias. Essas operações não corrigem automaticamente valores inválidos; seus domínios serão examinados adiante.

In [242]:
# Padronizar os nomes das espécies
df['SPECIES'] = df['SPECIES'].str.split().str[0]
df['SPECIES'].value_counts()

SPECIES
Adelie       152
Gentoo       124
Chinstrap     68
Name: count, dtype: int64

In [243]:
# Padronizar as categorias de sexo
df['SEX'] = df['SEX'].str.lower()
df['SEX'].value_counts()

SEX
male      168
female    165
Name: count, dtype: int64

### 7.3 Conversão de tipos

Espécie, região, ilha e sexo são variáveis qualitativas nominais. As quatro medidas corporais são quantitativas contínuas, mesmo quando registradas como inteiros. A data é temporal; extrair o ano não a transforma em uma contagem.

In [244]:
# Conversão de tipos qualitativos nominais
cols = df.columns.to_list()
cols_cat = ['SPECIES', 'REGION', 'ISLAND', 'SEX']
for coluna in cols_cat:
    df[coluna] = df[coluna].astype('category')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   SPECIES         344 non-null    category
 1   REGION          344 non-null    category
 2   ISLAND          344 non-null    category
 3   DATE_EGG        344 non-null    str     
 4   CULMEN_LENGTH   342 non-null    float64 
 5   CULMEN_DEPTH    342 non-null    float64 
 6   FLIPPER_LENGTH  342 non-null    float64 
 7   BODY_MASS       342 non-null    float64 
 8   SEX             333 non-null    category
 9   ID              344 non-null    str     
dtypes: category(4), float64(4), str(2)
memory usage: 17.7 KB


In [245]:
# Conversão para tipo quantitativos contínuos
cols_num = ['CULMEN_LENGTH', 'CULMEN_DEPTH', 'FLIPPER_LENGTH', 'BODY_MASS']
for coluna in cols_num:
    df[coluna] = df[coluna].astype('float')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   SPECIES         344 non-null    category
 1   REGION          344 non-null    category
 2   ISLAND          344 non-null    category
 3   DATE_EGG        344 non-null    str     
 4   CULMEN_LENGTH   342 non-null    float64 
 5   CULMEN_DEPTH    342 non-null    float64 
 6   FLIPPER_LENGTH  342 non-null    float64 
 7   BODY_MASS       342 non-null    float64 
 8   SEX             333 non-null    category
 9   ID              344 non-null    str     
dtypes: category(4), float64(4), str(2)
memory usage: 17.7 KB


In [246]:
# Converter a data e extrair o ano
df['DATE_EGG'] = pd.to_datetime(df['DATE_EGG'], format='%Y-%m-%d')
df['YEAR'] = df['DATE_EGG'].dt.year
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   SPECIES         344 non-null    category      
 1   REGION          344 non-null    category      
 2   ISLAND          344 non-null    category      
 3   DATE_EGG        344 non-null    datetime64[us]
 4   CULMEN_LENGTH   342 non-null    float64       
 5   CULMEN_DEPTH    342 non-null    float64       
 6   FLIPPER_LENGTH  342 non-null    float64       
 7   BODY_MASS       342 non-null    float64       
 8   SEX             333 non-null    category      
 9   ID              344 non-null    str           
 10  YEAR            344 non-null    int32         
dtypes: category(4), datetime64[us](1), float64(4), int32(1), str(1)
memory usage: 19.0 KB


O ano informa quando a observação ocorreu. Já o **número de observações em cada ano** é uma contagem quantitativa discreta: assume valores inteiros não negativos. Na saída abaixo, o índice mostra o ano e os valores mostram as contagens.

In [247]:
# Contar as observações de cada ano
df['YEAR'].value_counts().sort_index()

YEAR
2007    110
2008    114
2009    120
Name: count, dtype: int64

A região pode ser retirada desta cópia quando contém um único valor e não distingue as observações. Sua remoção não altera a base bruta.

In [248]:
# Conferir os valores da região antes de remover a coluna
df['REGION'].value_counts(dropna=False)

REGION
Anvers    344
Name: count, dtype: int64

In [249]:
# Removendo uma coluna
df = df.drop(columns='REGION')

df.head()

,SPECIES,ISLAND,DATE_EGG,CULMEN_LENGTH,CULMEN_DEPTH,FLIPPER_LENGTH,BODY_MASS,SEX,ID,YEAR
0,Adelie,Torgersen,2007-11-11,39.1,18.7,181.0,3750.0,male,PAL0708_N1A1,2007
1,Adelie,Torgersen,2007-11-11,39.5,17.4,186.0,3800.0,female,PAL0708_N1A2,2007
2,Adelie,Torgersen,2007-11-16,40.3,18.0,195.0,3250.0,female,PAL0708_N2A1,2007
3,Adelie,Torgersen,2007-11-16,NaN,NaN,NaN,NaN,NaN,PAL0708_N2A2,2007
4,Adelie,Torgersen,2007-11-16,36.7,19.3,193.0,3450.0,female,PAL0708_N3A1,2007


### 7.4 Completude e validade

Verifique valores ausentes, registros com espécie, massa e sexo preenchidos, categorias de sexo e medidas menores ou iguais a zero. Preenchimento não garante validade.

In [250]:
# Contar os valores ausentes em cada coluna
df.isna().sum()

SPECIES            0
ISLAND             0
DATE_EGG           0
CULMEN_LENGTH      2
CULMEN_DEPTH       2
FLIPPER_LENGTH     2
BODY_MASS          2
SEX               11
ID                 0
YEAR               0
dtype: int64

In [251]:
# Conferir as categorias de sexo e suas contagens
df['SEX'].value_counts(dropna=False)

SEX
male      168
female    165
NaN        11
Name: count, dtype: int64

In [252]:
# Contar as observações com espécie, massa e sexo preenchidos
completos = df['SPECIES'].notna() & df['BODY_MASS'].notna() & df['SEX'].notna()
completos.sum()

np.int64(333)

In [253]:
# Contar as medidas menores ou iguais a zero
(df[cols_num] <= 0).sum()

CULMEN_LENGTH     0
CULMEN_DEPTH      0
FLIPPER_LENGTH    0
BODY_MASS         0
dtype: int64

### 7.5 Unicidade e chaves

Uma chave identifica cada registro. A coluna `ID`, criada com a campanha e o identificador individual, deve ter um valor diferente em cada linha. Vamos verificar se esse identificador se repete.

Se houver repetições, examine os registros antes de decidir qual remover. Um mesmo identificador pode aparecer em linhas com informações diferentes (MCKINNEY, 2017, pp. 215–217).

In [254]:
# Verificar se o novo identificador possui repetições
df['ID'].duplicated().sum()

np.int64(0)

### 7.6 Exportação

Salvar os dados em um arquivo CSV. O argumento `index=False` evita gravar o índice como uma coluna adicional.

In [255]:
# Organizar as colunas pelas posições, contadas a partir de zero
ordem = [8, 0, 1, 7, 2, 9, 3, 4, 5, 6]
df = df.iloc[:, ordem]
df.head()

,ID,SPECIES,ISLAND,SEX,DATE_EGG,YEAR,CULMEN_LENGTH,CULMEN_DEPTH,FLIPPER_LENGTH,BODY_MASS
0,PAL0708_N1A1,Adelie,Torgersen,male,2007-11-11,2007,39.1,18.7,181.0,3750.0
1,PAL0708_N1A2,Adelie,Torgersen,female,2007-11-11,2007,39.5,17.4,186.0,3800.0
2,PAL0708_N2A1,Adelie,Torgersen,female,2007-11-16,2007,40.3,18.0,195.0,3250.0
3,PAL0708_N2A2,Adelie,Torgersen,NaN,2007-11-16,2007,NaN,NaN,NaN,NaN
4,PAL0708_N3A1,Adelie,Torgersen,female,2007-11-16,2007,36.7,19.3,193.0,3450.0


In [256]:
# Exportando o dado
df.to_csv('../../data/processed/penguins.csv', index=False)

## 8. Estudo e exercícios



### 8.1 Materiais didáticos

- MORETTIN; BUSSAB (2010), seção 2.1, páginas PDF 26–27: tipos de variáveis.
- BARBETTA; REIS; BORNIA (2010), seção 3.1, páginas PDF 52–53: dados, casos, variáveis, unidades e domínios.
- PINHEIRO et al. (2009), seção 1.3, páginas PDF 22–23: observações, variáveis e classificação por natureza estatística.
- BRUCE; BRUCE (2019), seção Elementos de Dados Estruturados, páginas PDF 23–26: tipos de dados e representação computacional.
- *Introdução à estatística para ciência de dados*, seção 1.2, páginas PDF 22–27: diagnóstico e pré-processamento.
- MCKINNEY (2017), seção 7.1, páginas PDF 209–214: dados ausentes.
- MCKINNEY (2017), seção 7.2, páginas PDF 215–228: transformações e duplicatas.

### 8.2 Exercícios indicados

No [Banco de questões e provas 2026.2](../../apostila/banco_questoes_provas_2026_2.pdf):

- questões 11 e 12, página 10;
- questões 15 e 17, página 11.

Ao responder, justifique a classificação pela natureza da variável e pelo processo de obtenção. Não use apenas o formato visual dos valores ou o tipo indicado pelo software.

### 8.3 Atividade prática

Em células separadas, usando a base Palmer Penguins:

1. Conte os valores ausentes de cada coluna e as observações com espécie, massa e sexo preenchidos.
2. Crie `ID` combinando campanha e identificador individual. Verifique se há repetições e explique o resultado.
3. Organize as colunas com `ID` na primeira posição e as medidas corporais ao final.
4. Exporte a cópia organizada para CSV, sem salvar o índice. Preserve o arquivo bruto.

## 9. Referências

- BARBETTA, Pedro Alberto; REIS, Marcelo Menezes; BORNIA, Antonio Cezar. *Estatística para cursos de engenharia e informática*. 3. ed. São Paulo: Atlas, 2010.
- BRUCE, Peter; BRUCE, Andrew. *Estatística prática para cientistas de dados: 50 conceitos essenciais*. Rio de Janeiro: Alta Books, 2019.
- *Introdução à estatística para ciência de dados: da exploração dos dados à experimentação contínua com exemplos de código em Python e R*. São Paulo: Casa do Código, 2025.
- MCKINNEY, Wes. *Python for Data Analysis: Data Wrangling with pandas, NumPy, and IPython*. 2. ed. Sebastopol: O'Reilly Media, 2017.
- MORETTIN, Pedro A.; BUSSAB, Wilton O. *Estatística básica*. 6. ed. São Paulo: Saraiva, 2010.
- PINHEIRO, João Ismael D. et al. *Estatística básica: a arte de trabalhar com dados*. Rio de Janeiro: Elsevier, 2009.